# Session 1 - Multi-Source Data Fusion

## OBJECTIVE

> Five defective data sources will be converted into one audit-ready AI modeling table.

Measurable targets:
- 840 unique records
- 0 duplicate keys
- 0 missing model cells
- 2 future fields quarantined

**Primary artifact:** AI Data Fusion Inspector

## SYSTEM MAP

`RAW SOURCES -> DATA CONTRACT -> QUALITY CONTROL -> SAFE FUSION -> FEATURES -> VALIDATION -> MODEL`

## WHY THIS MATTERS

### SYSTEM A
- Accuracy: 80%
- Future-only data: NO
- Duplicate audit: PASS
- Safe-join contract: PASS

### SYSTEM B
- Accuracy: 92%
- Future-only data: YES
- Duplicate audit: UNKNOWN
- Safe-join contract: UNKNOWN

**SYSTEM A is deployable. SYSTEM B is invalid despite the stronger score.**

> The model is not the AI system.

## 1. Path setup and imports

In [20]:
import sys
from pathlib import Path

# Find project root by searching upward for src/
current = Path.cwd()
for parent in [current] + list(current.parents):
    if (parent / "src").exists():
        root = parent
        break
else:
    root = current

if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

DATA_DIR = root / "src" / "data"
print(f"Project root: {root}")
print(f"Data dir: {DATA_DIR}")

Project root: c:\Users\Metwalli\Desktop\summer\Advanced_AI_Practical\AASTMT_Advanced_Practical_AI_V2
Data dir: c:\Users\Metwalli\Desktop\summer\Advanced_AI_Practical\AASTMT_Advanced_Practical_AI_V2\src\data


In [21]:
# Define some constants and constrains/Expected ranges for all data.
import numpy as np
import pandas as pd

RANDOM_STATE = 42
KEYS = ["region_id", "year", "month"]
CLASS_NAMES = np.array(["LOW", "MEDIUM", "HIGH"])

RAW_FEATURES = [
    "temperature_c", "rainfall_mm", "humidity_pct",
    "soil_moisture", "soil_ph", "vegetation_index",
    "disease_cases", "pest_reports",
    "irrigation_capacity", "vulnerability_index",
]

ENGINEERED_FEATURES = [
    "drought_index", "heat_stress", "disease_pressure",
    "water_balance", "season_sin", "season_cos",
]

MODEL_FEATURES = RAW_FEATURES + ENGINEERED_FEATURES
PROHIBITED_FEATURES = ["future_loss_index", "response_action_code"]
# Why should I put Valid Ranges and only for Raw Features?
VALID_RANGES = {
    "temperature_c": (-15.0, 60.0),
    "rainfall_mm": (0.0, 500.0),
    "humidity_pct": (0.0, 100.0),
    "soil_moisture": (0.0, 1.0),
    "soil_ph": (3.0, 10.0),
    "vegetation_index": (0.0, 1.0),
    "disease_cases": (0.0, 5000.0),
    "pest_reports": (0.0, 1000.0),
    "irrigation_capacity": (0.0, 1.0),
    "vulnerability_index": (0.0, 1.0),
}

print("Constants defined")
print(f"Expected observations: 10 regions x 7 years x 12 months = {10 * 7 * 12}")

Constants defined
Expected observations: 10 regions x 7 years x 12 months = 840


## 2. Generate or load raw sources

In [22]:
# Generate deterministic synthetic data
rng = np.random.default_rng(RANDOM_STATE)

def make_region_profiles():
    regions = pd.DataFrame()
    regions["region_id"] = [f"R{str(i).zfill(2)}" for i in range(1, 11)]
    regions["region_name"] = [
        "Delta", "Nile Valley", "Coastal", "Upper Egypt", "Sinai",
        "Western Desert", "Eastern Desert", "Fayoum", "Red Sea", "Matrouh",
    ]
    regions["irrigation_capacity"] = rng.uniform(0.3, 0.9, 10).round(4)
    regions["vulnerability_index"] = rng.uniform(0.1, 0.8, 10).round(4)
    climate = ["arid", "semi_arid", "mediterranean", "arid", "arid",
               "hyper_arid", "arid", "semi_arid", "arid", "mediterranean"]
    regions["climate_band"] = climate
    return regions

In [23]:
# Generation of data

raw_dir = DATA_DIR / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

# Check if sources already exist
if (raw_dir / "weather.csv").exists():
    weather = pd.read_csv(raw_dir / "weather.csv")
    soil = pd.read_csv(raw_dir / "soil.csv")
    disease = pd.read_csv(raw_dir / "disease.csv")
    regions = pd.read_csv(raw_dir / "regions.csv")
    outcomes = pd.read_csv(raw_dir / "outcomes.csv")
    print("Loaded existing sources")
else:
    # Generate and save
    from reliable_ai.data import generate_raw_sources
    generate_raw_sources(DATA_DIR)
    weather = pd.read_csv(raw_dir / "weather.csv")
    soil = pd.read_csv(raw_dir / "soil.csv")
    disease = pd.read_csv(raw_dir / "disease.csv")
    regions = pd.read_csv(raw_dir / "regions.csv")
    outcomes = pd.read_csv(raw_dir / "outcomes.csv")
    print("Generated and saved sources")

Loaded existing sources


## 3. Source audit

In [31]:
sources = {
    "weather": weather,
    "soil": soil,
    "disease": disease,
    "regions": regions,
    "outcomes": outcomes,
}

for name, frame in sources.items():
    print(f"\n{name.upper()}:")
    print(f"  Shape: {frame.shape}")
    print(f"  Missing cells: {frame.isna().sum().sum()}")
    print(frame.head(3).to_string(index=False))


WEATHER:
  Shape: (841, 6)
  Missing cells: 1
region_id  year  month  temperature_c  rainfall_mm  humidity_pct
      R01  2019      1           22.9         36.7          77.5
      R01  2019      2           27.3         15.7          54.3
      R01  2019      3           26.7         30.3          59.8

SOIL:
  Shape: (840, 6)
  Missing cells: 1
region_id  year  month  soil_moisture  soil_ph  vegetation_index
      R01  2019      1         0.5474     7.49            0.6286
      R01  2019      2         0.2839     7.55            0.6981
      R01  2019      3         0.3916     6.41            0.6832

DISEASE:
  Shape: (841, 5)
  Missing cells: 1
region_id  year  month  disease_cases  pest_reports
      R01  2019      1           57.6          20.3
      R01  2019      2           68.6          13.5
      R01  2019      3           33.1          37.9

REGIONS:
  Shape: (10, 5)
  Missing cells: 0
region_id region_name  irrigation_capacity  vulnerability_index  climate_band
      R01 

In [32]:
# We need to remove duplicates/Fill Missing Values/and Checkout Outliers and all assessments
from reliable_ai.data import audit_sources
audit = audit_sources(sources)
print(audit.to_string(index=False))

  Source  Rows  Missing cells  Duplicate composite keys
 weather   841              1                         1
    soil   840              1                         0
 disease   841              1                         1
 regions    10              0                         0
outcomes   840              0                         0


## 4. PREDICT BEFORE RUNNING

> Weather contains 841 rows, but 840 unique region-month keys are expected. What is the most likely reason?

Record your hypothesis:

*Your hypothesis:* A duplicate composite key or an unexpected region-month record exists.

## 5. Duplicate composite-key check

In [7]:
# TODO: Count duplicated region-month keys in weather
weather_duplicate_count = ...
print(f"Weather duplicate keys: {weather_duplicate_count}")

Weather duplicate keys: Ellipsis


In [8]:
# TODO: Count duplicated region-month keys in disease
disease_duplicate_count = ...
print(f"Disease duplicate keys: {disease_duplicate_count}")

Disease duplicate keys: Ellipsis


In [9]:
print(f"Soil duplicate keys: {soil.duplicated(subset=KEYS).sum()}")
print(f"Outcomes duplicate keys: {outcomes.duplicated(subset=KEYS).sum()}")

Soil duplicate keys: 0
Outcomes duplicate keys: 0


## 6. BREAK IT - Strict merge before cleaning

Attempt a strict one-to-one merge before duplicate repair.

In [10]:
try:
    broken = pd.merge(weather, soil, on=KEYS, validate="one_to_one")
    print("Merge succeeded (unexpected)")
except Exception as e:
    print(f"Merge failed: {type(e).__name__}: {e}")
    print("The merge contract blocked silent row multiplication.")

Merge failed: MergeError: Merge keys are not unique in left dataset; not a one-to-one merge
Duplicates in left:
 region_id  year  month
      R04  2023      1 ...
The merge contract blocked silent row multiplication.


## 7. FIX IT - Clean and repair

### 7a. Resolve duplicates

In [11]:
from reliable_ai.data import aggregate_duplicates

weather_clean = aggregate_duplicates(weather)
disease_clean = aggregate_duplicates(disease)
soil_clean = aggregate_duplicates(soil)

print(f"Weather after dedup: {len(weather_clean)}")
print(f"Disease after dedup: {len(disease_clean)}")

Weather after dedup: 840
Disease after dedup: 840


In [12]:
# Assertion checkpoint
assert weather_clean.duplicated(subset=KEYS).sum() == 0
assert disease_clean.duplicated(subset=KEYS).sum() == 0
print("Duplicate keys resolved")

Duplicate keys resolved


### 7b. Invalidate impossible ranges

> Invalid is not low.

In [13]:
# Inspect the invalid values before conversion
bad_rain = weather_clean[weather_clean["rainfall_mm"] < 0]
bad_ph = soil_clean[soil_clean["soil_ph"] > 10]
print(f"Rows with negative rainfall: {len(bad_rain)}")
print(f"Rows with pH > 10: {len(bad_ph)}")

Rows with negative rainfall: 1
Rows with pH > 10: 1


In [14]:
# TODO: Convert values outside VALID_RANGES to NaN
def invalidate_ranges(frame, ranges):
    result = frame.copy()
    for col, (lo, hi) in ranges.items():
        ...
    return result

weather_clean = invalidate_ranges(weather_clean, VALID_RANGES)
soil_clean = invalidate_ranges(soil_clean, VALID_RANGES)
disease_clean = invalidate_ranges(disease_clean, VALID_RANGES)
print(f"Invalid values converted to NaN")

Invalid values converted to NaN


### 7c. Region-aware temporal interpolation

> A missing monthly measurement in a region should first use the temporal context of that region, not the median behavior of every region.

In [15]:
from reliable_ai.data import temporal_impute

weather_clean = temporal_impute(weather_clean)
soil_clean = temporal_impute(soil_clean)
disease_clean = temporal_impute(disease_clean)

print(f"Weather missing after impute: {weather_clean.isna().sum().sum()}")
print(f"Soil missing after impute: {soil_clean.isna().sum().sum()}")
print(f"Disease missing after impute: {disease_clean.isna().sum().sum()}")

Weather missing after impute: 0
Soil missing after impute: 0
Disease missing after impute: 0


## 8. Safe fusion with explicit contracts

In [16]:
fused = pd.merge(weather_clean, soil_clean, on=KEYS, how="inner", validate="one_to_one")
fused = pd.merge(fused, disease_clean, on=KEYS, how="inner", validate="one_to_one")
fused = pd.merge(fused, regions, on="region_id", how="inner", validate="many_to_one")
fused = pd.merge(fused, outcomes, on=KEYS, how="inner", validate="one_to_one")

fused = fused.sort_values(KEYS).reset_index(drop=True)
print(f"Fused table: {fused.shape[0]} rows x {fused.shape[1]} columns")

Fused table: 840 rows x 18 columns


In [17]:
# Assertion checkpoint
assert fused.duplicated(subset=KEYS).sum() == 0
print("No duplicate keys after fusion")

missing_in_model = fused[MODEL_FEATURES].isna().sum().sum()
assert missing_in_model == 0, f"{missing_in_model} missing values remain"
print(f"Missing model cells: {missing_in_model}")

No duplicate keys after fusion


KeyError: "['drought_index', 'heat_stress', 'disease_pressure', 'water_balance', 'season_sin', 'season_cos'] not in index"

## 9. Feature engineering

> December and January are neighbors.

In [ ]:
def clip01(values):
    return np.clip(values, 0.0, 1.0)

fused["drought_index"] = clip01(1.0 - fused["rainfall_mm"] / 115.0)
fused["heat_stress"] = clip01((fused["temperature_c"] - 27.0) / 12.0)
fused["disease_pressure"] = clip01(fused["disease_cases"] / 115.0)
fused["water_balance"] = clip01(
    0.55 * fused["soil_moisture"]
    + 0.30 * fused["irrigation_capacity"]
    + 0.15 * (1.0 - fused["drought_index"])
)

angle = 2.0 * np.pi * (fused["month"] - 1) / 12.0
fused["season_sin"] = np.sin(angle)
fused["season_cos"] = np.cos(angle)

print(f"Engineered features added: {ENGINEERED_FEATURES}")

In [ ]:
# Verify all model features are present and complete
assert all(f in fused.columns for f in MODEL_FEATURES)
print(f"All {len(MODEL_FEATURES)} model features present")
print(f"Prohibited features in model: {set(PROHIBITED_FEATURES) & set(MODEL_FEATURES)}")

## 10. Feature registry

In [ ]:
rows = []
for col in MODEL_FEATURES:
    rows.append({"feature": col, "role": "approved_model_feature"})
for col in PROHIBITED_FEATURES:
    rows.append({"feature": col, "role": "prohibited_post_outcome"})
rows.append({"feature": "risk_label", "role": "target"})
for col in KEYS + ["region_name", "climate_band"]:
    rows.append({"feature": col, "role": "identifier_or_context"})

feature_registry = pd.DataFrame(rows)
print(feature_registry.to_string(index=False))

## 11. Export artifacts

In [ ]:
processed_dir = DATA_DIR / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

fused.to_csv(processed_dir / "modeling_table.csv", index=False)
audit.to_csv(processed_dir / "source_audit.csv", index=False)
feature_registry.to_csv(processed_dir / "feature_registry.csv", index=False)
print("Artifacts exported")

## 12. EVIDENCE - AI Data Fusion Inspector

In [ ]:
n_approved = len(MODEL_FEATURES)
n_prohibited = len(PROHIBITED_FEATURES)
final_rows = len(fused)
dup_keys = fused.duplicated(subset=KEYS).sum()
missing = fused[MODEL_FEATURES].isna().sum().sum()

print("=" * 60)
print("AI DATA FUSION INSPECTOR")
print("=" * 60)
print(f"{"Expected observations":30s}: 840")
print(f"{"Final observations":30s}: {final_rows}")
print(f"{"Duplicate keys":30s}: {dup_keys}")
print(f"{"Missing model cells":30s}: {missing}")
print(f"{"Approved model features":30s}: {n_approved}")
print(f"{"Prohibited future fields":30s}: {n_prohibited}")
print(f"{"Fusion status":30s}: PASSED")

## 13. ASSIGNMENT

**Assignment 01 - Data Contract Recovery** extends the same pattern to a new defect set.

See `assignments/Assignment_01_Data_Contract_Recovery.pdf`.

## SESSION RESULT

> A trusted 840-row modeling table now exists.

The next question becomes: Can a model trained on earlier years genuinely generalize to a later year without cheating?